# 03 | Revised Causal Diagnostics

Audit the credibility of the intervention prototype using a later 78-feature cohort pipeline. The analysis is observational; treatment is inferred from prices and payments. Removing directly treatment-revealing variables reduces one source of leakage, but does not establish that all remaining covariates precede treatment.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Run from the repository root or notebooks/ directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
PARQ = DATA / "parquet"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import log_loss
from scipy.stats import norm
import matplotlib.pyplot as plt

## Load data and exclude direct payment signals

In [ ]:
def load_members(cutoff):

    m = pd.read_parquet(f'{PARQ}/members_v3.parquet',
                        columns=['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time'])
    m['msno'] = m['msno'].astype(str)
    m['bd'] = np.where((m['bd'] < 1) | (m['bd'] > 100), np.nan, m['bd'])
    m['gender_enc'] = m['gender'].map({'male': 1, 'female': 0})
    cut = pd.to_datetime(str(cutoff), format='%Y%m%d')
    reg = pd.to_datetime(m['registration_init_time'].astype(str), format='%Y%m%d', errors='coerce')
    m['tenure_days'] = (cut - reg).dt.days
    return m[['msno', 'city', 'bd', 'gender_enc', 'registered_via', 'tenure_days']]

feats = pd.read_parquet(f'{PARQ}/top_features_train.parquet')
feats['msno'] = feats['msno'].astype(str)
labels = pd.read_parquet(f'{PARQ}/train_v2.parquet', columns=['msno', 'is_churn'])
labels['msno'] = labels['msno'].astype(str)

d = feats.merge(load_members(20170131), on='msno', how='left').merge(labels, on='msno', how='inner')


d['grp'] = np.where((d['last_paid'] == 0) & (d['last_price'] > 0), 'free',
            np.where(d['last_discount'] > 0, 'discount', 'control'))


LEAK = ['last_paid', 'last_discount', 'last_discount_rate', 'last_price_per_day', 'mean_paid',
        'std_paid', 'mean_discount', 'max_discount', 'last5_mean_discount', 'last_price_change', 'last_price']
covars = [c for c in feats.columns if c not in LEAK and c != 'msno'] + ['city', 'bd', 'gender_enc', 'tenure_days']
X = d[covars].astype('float32').replace([np.inf, -np.inf], np.nan).fillna(0).values
y = d['is_churn'].values

print('Observed proxy group counts:', d['grp'].value_counts().to_dict())

## Matching and observed balance

The original diagnostic uses logistic propensity models, nearest-neighbor matching with replacement, and a caliper of 0.2 times the standard deviation of the logit propensity. The original variable name `ATE` is retained internally; the reported contrast is a matched control-minus-treated churn-rate difference, targeting matched treated users rather than the population ATE. The interval uses an independent-proportions approximation; it ignores matching dependence and propensity estimation uncertainty. It is not a valid definitive causal confidence interval.

In [ ]:
def psm_ate(treat):

    is_t = (d['grp'] == treat).values
    is_c = (d['grp'] == 'control').values
    Xt, Xc, yt, yc = X[is_t], X[is_c], y[is_t], y[is_c]


    Xtc = np.vstack([Xt, Xc])
    treat_flag = np.r_[np.ones(len(Xt)), np.zeros(len(Xc))]
    scaler = StandardScaler().fit(Xtc)
    ps = LogisticRegression(C=1.0, max_iter=500).fit(scaler.transform(Xtc), treat_flag).predict_proba(scaler.transform(Xtc))[:, 1]
    logit = np.log(np.clip(ps, 1e-6, 1 - 1e-6) / (1 - np.clip(ps, 1e-6, 1 - 1e-6)))


    caliper = 0.2 * logit.std()
    nn = NearestNeighbors(n_neighbors=1).fit(logit[len(Xt):].reshape(-1, 1))
    dist, idx = nn.kneighbors(logit[:len(Xt)].reshape(-1, 1))
    keep = dist.ravel() <= caliper
    matched_c = idx.ravel()[keep]


    p_t, p_c = yt[keep].mean(), yc[matched_c].mean()
    n = keep.sum()
    se = np.sqrt(p_t * (1 - p_t) / n + p_c * (1 - p_c) / n)
    smd = lambda a, b: np.abs((a.mean(0) - b.mean(0)) / np.sqrt((a.var(0) + b.var(0)) / 2 + 1e-9))
    return dict(n=int(is_t.sum()), matched_n=int(n), ATE=p_c - p_t, CI_low=p_c - p_t - 1.96 * se, CI_high=p_c - p_t + 1.96 * se,
                SMD_before=smd(Xt, Xc).max(), SMD_after=smd(Xt[keep], Xc[matched_c]).max())

ate = {t: psm_ate(t) for t in ['discount', 'free']}
pd.DataFrame(ate).T.rename(columns={'ATE':'matched_churn_difference','CI_low':'naive_CI_low','CI_high':'naive_CI_high'}).round(4)

## Historical matched contrasts and balance (recorded, not rerun)

| Proxy treatment | Observed count | Max SMD before | Max SMD after | Matched churn-rate difference (pp) |
|---|---:|---:|---:|---:|
| Discount | 654 | 5.5992 | 0.3341 | +3.37 |
| Free | 1,998 | 1.1396 | 0.3677 | -3.61 |

The post-matching imbalance remains above the commonly used 0.1 heuristic. Neither contrast establishes the sign of a causal effect. The subsidy proxies represent only approximately 0.28% of the training cohort.

## Expected net benefit sensitivity

This section applies the original aggregate matched differences to hypothetical retained-month and cost assumptions. It illustrates sensitivity, not profitability. A negative estimate does not establish that free trials are always unprofitable.

In [ ]:
ATE_D, COST_D = ate['discount']['ATE'], 35.0
ATE_F, COST_F = ate['free']['ATE'], 149.0
MONTHLY = 149.0

enb = pd.DataFrame({
    'retained_months': [1, 3, 6, 12],
    'discount_ENB_per_user': [round(ATE_D * MONTHLY * m - COST_D, 1) for m in [1, 3, 6, 12]],
    'free_ENB_per_user': [round(ATE_F * MONTHLY * m - COST_F, 1) for m in [1, 3, 6, 12]],
})
enb

## Descriptive group profiles and listening segments

These patterns suggest possible selection mechanisms but cannot identify the actual assignment policy or who is persuadable. Small segments should be interpreted cautiously.

In [ ]:
profile = d.groupby('grp').agg(
    users=('is_churn', 'size'),
    churn_rate=('is_churn', 'mean'),
    mean_transactions=('n_txn', 'mean'),
    auto_renew_rate=('autorenew_rate', 'mean'),
    mean_tenure_days=('tenure_days', 'mean'),
).round({'churn_rate': 4, 'mean_transactions': 1, 'auto_renew_rate': 3, 'mean_tenure_days': 0})
profile

In [ ]:
free = d[d['grp'] == 'free'].copy()
free['listening_recency'] = np.where(free['last_ul_days_s'] <= 14, 'active_within_14_days', 'inactive_over_14_days')
free['listening_depth'] = np.where(free['ul_all_active_days'] >= 300, 'heavy',
                   np.where(free['ul_all_active_days'] >= 100, 'medium', 'light'))
seg = (free.groupby(['listening_recency', 'listening_depth'])
           .agg(users=('is_churn', 'size'), churn_rate=('is_churn', 'mean'))
           .round(4).sort_values('churn_rate'))
seg

## S-, T-, and X-Learners: out-of-fold Qini

The original evaluation samples 100,000 controls per contrast, retains all treated rows, and uses five folds stratified by treatment and observed outcome. Qini is an observational ranking diagnostic here; it does not resolve confounding or validate individual treatment effects.

In [ ]:
from sklift.models import SoloModel, TwoModels
from sklift.metrics import qini_auc_score
from lightgbm import LGBMClassifier, LGBMRegressor

new_clf = lambda: LGBMClassifier(n_estimators=250, num_leaves=31, learning_rate=0.05, n_jobs=4, verbose=-1)
new_reg = lambda: LGBMRegressor(n_estimators=250, num_leaves=31, learning_rate=0.05, n_jobs=4, verbose=-1)

qini_result = {}
for treat in ['discount', 'free']:

    if treat == 'discount':
        w_all = np.where((d['last_paid'] > 0) & (d['last_paid'] < d['last_price']), 1,
                         np.where(d['last_paid'] >= d['last_price'], 0, -1))
    else:
        w_all = np.where((d['last_paid'] == 0) & (d['last_price'] > 0), 1,
                         np.where(d['last_paid'] >= d['last_price'], 0, -1))

    ctrl_idx = np.where(w_all == 0)[0]
    np.random.seed(42)
    keep_ctrl = np.random.choice(ctrl_idx, 100000, replace=False)
    idx = np.r_[keep_ctrl, np.where(w_all == 1)[0]]
    Xs, ws, ys = X[idx], w_all[idx].astype(int), (1 - y[idx]).astype(int)

    oof = {k: np.zeros(len(ys)) for k in ['S', 'T', 'X']}
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=1).split(Xs, ws * 2 + ys):

        s = SoloModel(new_clf()); s.fit(Xs[tr], ys[tr], ws[tr]); oof['S'][va] = s.predict(Xs[va])

        t = TwoModels(new_clf(), new_clf(), method='vanilla'); t.fit(Xs[tr], ys[tr], ws[tr]); oof['T'][va] = t.predict(Xs[va])

        m0 = new_clf().fit(Xs[tr][ws[tr] == 0], ys[tr][ws[tr] == 0])
        m1 = new_clf().fit(Xs[tr][ws[tr] == 1], ys[tr][ws[tr] == 1])
        D1 = ys[tr][ws[tr] == 1] - m0.predict_proba(Xs[tr][ws[tr] == 1])[:, 1]
        D0 = m1.predict_proba(Xs[tr][ws[tr] == 0])[:, 1] - ys[tr][ws[tr] == 0]
        tau1 = new_reg().fit(Xs[tr][ws[tr] == 1], D1)
        tau0 = new_reg().fit(Xs[tr][ws[tr] == 0], D0)
        sc = StandardScaler().fit(Xs[tr])
        g = LogisticRegression(max_iter=300).fit(sc.transform(Xs[tr]), ws[tr]).predict_proba(sc.transform(Xs[va]))[:, 1]
        oof['X'][va] = g * tau0.predict(Xs[va]) + (1 - g) * tau1.predict(Xs[va])
    qini_result[treat] = {k: round(qini_auc_score(ys, oof[k], ws), 4) for k in oof}

pd.DataFrame(qini_result)

## Historical Qini scores (recorded, not rerun)

| Learner | Discount | Free |
|---|---:|---:|
| S | 0.0576 | 0.0608 |
| T | 0.0574 | 0.0579 |
| X | 0.0198 | 0.0164 |

The combination of sparse proxy treatment, residual imbalance, and limited ranking evidence does not justify direct deployment. The next step is a controlled experiment.